# Clark-West (2007) Predictive Accuracy Analysis — Naïve vs. Each Model
### DAMO-699 Capstone Project | Group 5

Under the refined analytical objective (§3) and research question (§3.1), this study evaluates:
> **"Which modelling approach (Naïve Random Walk, ARIMA, VAR/VECM, Long Short-Term Memory [LSTM] Neural Network) most reliably forecasts near-term (1–20 day) changes in the Canadian 10y–2y yield spread for use in an operational decision?"**

To answer this question rigorously, each candidate model is compared **exclusively against the common Naïve Random Walk benchmark** ($\Delta s_{t+h} = 0$) using the **Clark-West (2007) adjusted MSPE test** with **Benjamini-Hochberg (1995) False Discovery Rate (FDR)** control.

---

## 1. Econometric Foundations: Why Clark-West (2007) Instead of Diebold-Mariano

### The Nested Model Estimation Noise Problem
Comparing the zero-change benchmark ($e_{1, t+h} = y_{t+h} - y_t$) against parametric and machine learning models ($e_{2, t+h} = y_{t+h} - \hat{y}_{2, t+h}$) involves **strictly nested models** (Clark & McCracken, 2001; Clark & West, 2006, 2007).

Under the null hypothesis ($H_0$) of equal population predictive accuracy (i.e. the extra features and parameters in Model 2 add zero true predictive power in population), Model 2 estimates parameters that are zero in population. In finite samples, parameter estimation introduces estimation noise of order $\mathcal{O}_p(T^{-1})$. This sample noise mechanically inflates Model 2's sample Mean Squared Prediction Error:
$$\mathbb{E}[\text{MSPE}_2] \approx \mathbb{E}[\text{MSPE}_1] + \mathcal{O}(T^{-1})$$

Consequently, the standard Diebold-Mariano loss differential $d_{t+h} = e_{1, t+h}^2 - e_{2, t+h}^2$ has a **negative expected value** under $H_0$, biasing standard DM tests toward finding that the Naïve benchmark "beats" the larger model spuriously.

### The Clark-West (2007) Correction
Clark & West (2007) resolve this asymptotic bias by adjusting the loss differential by the sample variance of the difference between the two models' predictions:
$$\hat{f}_{t+h} = e_{1, t+h}^2 - \left[ e_{2, t+h}^2 - (\hat{y}_{1, t+h} - \hat{y}_{2, t+h})^2 \right] = 2 e_{1, t+h}(\hat{y}_{2, t+h} - \hat{y}_{1, t+h})$$

where:
- $\text{MSPE}_1 = \frac{1}{N} \sum e_{1, t+h}^2$
- $\text{MSPE}_2 = \frac{1}{N} \sum e_{2, t+h}^2$
- $\text{adj} = \frac{1}{N} \sum (\hat{y}_{1, t+h} - \hat{y}_{2, t+h})^2$
- $\text{MSPE}_2^{\text{adj}} = \text{MSPE}_2 - \text{adj}$
- $\bar{f} = \text{MSPE}_1 - \text{MSPE}_2^{\text{adj}}$

The test statistic is computed by standardizing $\bar{f}$ with a calendar-safe Heteroskedasticity and Autocorrelation Consistent (HAC / Newey-West) variance estimator with Bartlett kernel and bandwidth $J = h - 1$ to account for multi-step forecast overlap, testing the one-sided alternative $H_A: \text{MSPE}_1 > \text{MSPE}_2$ ($p = 1 - \Phi(CW)$).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Ensure src/ is on the path
_bootstrap_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_bootstrap_root / "src"))

from project_paths import find_project_root
PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from model_comparison import (
    clark_west_test,
    run_clark_west_battery,
    apply_clark_west_fdr,
    plain_language_verdict_cw,
    HORIZONS,
    ALPHA,
    OUT_DIR,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print(f"Project root resolved: {PROJECT_ROOT}")

Project root resolved: C:\Users\User\OneDrive\Documentos\Master in data analytics\Capstone Project


## 2. Execute Primary 12-Test Clark-West Battery

The primary evaluation matrix comprises **4 model paradigms × 3 horizons = 12 hypothesis tests**:
1. **ARIMA-AIC** vs. Naïve ($h=1, 5, 20$)
2. **VAR-AIC** vs. Naïve ($h=1, 5, 20$)
3. **VECM (6-variable)** vs. Naïve ($h=1, 5, 20$)
4. **LSTM (Tuned)** vs. Naïve ($h=1, 5, 20$)

Companion BIC sensitivity arms (ARIMA-BIC, VAR-BIC) are scored separately to maintain transparent methodological sensitivity checks without inflating the primary 12-test family multiplicity.

In [2]:
primary_df, sensitivity_df = run_clark_west_battery(alpha=ALPHA)

# Save canonical result datasets to outputs/
primary_df.to_csv(OUT_DIR / "clark_west_test_results.csv", index=False)
sensitivity_df.to_csv(OUT_DIR / "clark_west_sensitivity_results.csv", index=False)

print("Primary 12-Test Matrix:")
display_cols = [
    "model", "horizon", "n_forecasts", "mspe_naive", "mspe_model",
    "cw_adjustment", "mspe_model_adj", "cw_stat", "cw_p_value",
    "cw_p_adj_global", "cw_p_adj_horizon"
]
print(primary_df[display_cols].to_string(index=False))

Primary 12-Test Matrix:
       model  horizon  n_forecasts  mspe_naive  mspe_model  cw_adjustment  mspe_model_adj  cw_stat  cw_p_value  cw_p_adj_global  cw_p_adj_horizon
   ARIMA-AIC        1          750    0.000841    0.000847       0.000006        0.000841   -0.080      0.5318           0.7872            0.8309
   ARIMA-AIC        5          750    0.003741    0.003828       0.000065        0.003764   -0.615      0.7306           0.7970            0.7306
   ARIMA-AIC       20          750    0.015639    0.016423       0.000605        0.015818   -0.402      0.6560           0.7872            0.6560
     VAR-AIC        1          750    0.000841    0.000887       0.000042        0.000846   -0.314      0.6232           0.7872            0.8309
     VAR-AIC        5          750    0.003741    0.004050       0.000297        0.003753   -0.123      0.5490           0.7872            0.7306
     VAR-AIC       20          750    0.015639    0.015832       0.000949        0.014883    0.970  

## 3. MSPE Decomposition and Parameter Estimation Noise Analysis

The table below isolates the exact contribution of the Clark-West adjustment term $(\hat{y}_1 - \hat{y}_2)^2$:
- $\text{MSPE}_{\text{model}} - \text{MSPE}_{\text{naive}}$: Raw MSPE differential (negative means model is raw-better).
- $\text{CW Adjustment}$: Estimation variance correction ($\|\hat{y}_1 - \hat{y}_2\|^2$).
- $\text{MSPE}_{\text{model}}^{\text{adj}} - \text{MSPE}_{\text{naive}}$: Adjusted MSPE differential ($-\bar{f}$).

In [3]:
decomp = primary_df.copy()
decomp["raw_diff"] = decomp["mspe_model"] - decomp["mspe_naive"]
decomp["adj_diff"] = decomp["mspe_model_adj"] - decomp["mspe_naive"]
decomp["adj_pct_reduction"] = (decomp["cw_adjustment"] / decomp["mspe_model"]) * 100

decomp_table = decomp[[
    "model", "horizon", "mspe_naive", "mspe_model", "raw_diff",
    "cw_adjustment", "adj_pct_reduction", "mspe_model_adj", "adj_diff", "cw_stat", "cw_p_value"
]]
print(decomp_table.to_string(index=False))

       model  horizon  mspe_naive  mspe_model  raw_diff  cw_adjustment  adj_pct_reduction  mspe_model_adj  adj_diff  cw_stat  cw_p_value
   ARIMA-AIC        1    0.000841    0.000847  0.000006       0.000006           0.708383        0.000841  0.000000   -0.080      0.5318
   ARIMA-AIC        5    0.003741    0.003828  0.000087       0.000065           1.698015        0.003764  0.000023   -0.615      0.7306
   ARIMA-AIC       20    0.015639    0.016423  0.000784       0.000605           3.683858        0.015818  0.000179   -0.402      0.6560
     VAR-AIC        1    0.000841    0.000887  0.000046       0.000042           4.735062        0.000846  0.000005   -0.314      0.6232
     VAR-AIC        5    0.003741    0.004050  0.000309       0.000297           7.333333        0.003753  0.000012   -0.123      0.5490
     VAR-AIC       20    0.015639    0.015832  0.000193       0.000949           5.994189        0.014883 -0.000756    0.970      0.1660
VECM (6-var)        1    0.000841    0.00

## 4. Multiplicity Control (Benjamini-Hochberg False Discovery Rate)

Across the 12 primary hypothesis tests, unadjusted $\alpha = 0.05$ creates a family-wise error risk ($1 - (1-0.05)^{12} \approx 46\%$). We apply **Benjamini-Hochberg (1995) FDR control** at two levels:
1. **Global 12-Test FDR (`cw_p_adj_global`)**: Evaluates the family of all 12 tests across models and horizons.
2. **Horizon-Stratified FDR (`cw_p_adj_horizon`)**: Evaluates families of 4 tests within each horizon $h \in \{1, 5, 20\}$, ensuring that high-variance noise at $h=20$ does not inflate discovery thresholds for short horizons ($h=1, 5$) (per Issue #81).

In [4]:
print("=== SUMMARY OF STATISTICAL DISCOVERIES BY HORIZON ===")
for h in HORIZONS:
    sub = primary_df[primary_df["horizon"] == h]
    print(f"\n--- Horizon {h} Day(s) ---")
    for _, r in sub.iterrows():
        print(f"  {r['model']:15s}: CW={r['cw_stat']:6.3f}, p_raw={r['cw_p_value']:6.4f}, q_glob={r['cw_p_adj_global']:6.4f}, q_horiz={r['cw_p_adj_horizon']:6.4f}")
        print(f"    Verdict (Raw)   : {r['verdict_raw']}")
        print(f"    Verdict (Horiz) : {r['verdict_fdr_horizon']}")

=== SUMMARY OF STATISTICAL DISCOVERIES BY HORIZON ===

--- Horizon 1 Day(s) ---
  ARIMA-AIC      : CW=-0.080, p_raw=0.5318, q_glob=0.7872, q_horiz=0.8309
    Verdict (Raw)   : No significant improvement: ARIMA-AIC does not beat Naïve under Clark-West (CW=-0.080, p=0.5318; MSPE_naive=0.00084, MSPE_model=0.00085)
    Verdict (Horiz) : No significant improvement: ARIMA-AIC does not beat Naïve under Clark-West (CW=-0.080, p=0.8309; MSPE_naive=0.00084, MSPE_model=0.00085)
  VAR-AIC        : CW=-0.314, p_raw=0.6232, q_glob=0.7872, q_horiz=0.8309
    Verdict (Raw)   : No significant improvement: VAR-AIC does not beat Naïve under Clark-West (CW=-0.314, p=0.6232; MSPE_naive=0.00084, MSPE_model=0.00089)
    Verdict (Horiz) : No significant improvement: VAR-AIC does not beat Naïve under Clark-West (CW=-0.314, p=0.8309; MSPE_naive=0.00084, MSPE_model=0.00089)
  VECM (6-var)   : CW= 0.452, p_raw=0.3257, q_glob=0.6514, q_horiz=0.8309
    Verdict (Raw)   : Positive point gain not statistically signif

## 5. Companion Sensitivity Checks (ARIMA-BIC & VAR-BIC)

To evaluate sensitivity to lag/order selection criteria, we assess the BIC-selected specifications against Naïve.

In [5]:
print("=== BIC SENSITIVITY BATTERY ===")
print(sensitivity_df[[
    "model", "horizon", "n_forecasts", "mspe_naive", "mspe_model",
    "cw_adjustment", "mspe_model_adj", "cw_stat", "cw_p_value", "cw_p_adj_global"
]].to_string(index=False))

=== BIC SENSITIVITY BATTERY ===
    model  horizon  n_forecasts  mspe_naive  mspe_model  cw_adjustment  mspe_model_adj  cw_stat  cw_p_value  cw_p_adj_global
ARIMA-BIC        1          750    0.000841    0.000842       0.000001        0.000841    0.197      0.4219           0.5286
ARIMA-BIC        5          750    0.003741    0.003764       0.000022        0.003742   -0.070      0.5280           0.5286
ARIMA-BIC       20          750    0.015639    0.016014       0.000358        0.015656   -0.048      0.5191           0.5286
  VAR-BIC        1          750    0.000841    0.000842       0.000001        0.000841    0.196      0.4224           0.5286
  VAR-BIC        5          750    0.003741    0.003764       0.000022        0.003742   -0.072      0.5286           0.5286
  VAR-BIC       20          750    0.015639    0.016011       0.000355        0.015656   -0.049      0.5196           0.5286


## 6. Synthesis, Econometric Discussion & Academic Framing

### 1. Daily Horizon ($h=1$): Martingale Property & Efficient Market Hypothesis (EMH)
Across all models (ARIMA, VAR, VECM, LSTM), test statistics at $h=1$ are non-significant ($CW \le 0.452$, $p \ge 0.3257$). Under Clark-West adjusted MSPE, no model beats the Naïve Random Walk. Rather than representing a statistical deficiency, this empirical result provides strong evidence of the **Martingale Property of Asset Prices** under the **Efficient Market Hypothesis** (Fama, 1970; Campbell, Lo, & MacKinlay, 1997; Duffee, 2002). At daily frequencies, yield spread changes reflect unforecastable information arrival and news innovations ($\mathbb{E}[\Delta s_{t+1} \mid \mathcal{I}_t] = 0$).

### 2. Multi-Step Horizon Dynamics ($h=5, 20$) & FDR Control
- **$h=5$ (Weekly)**: LSTM achieves $CW = 1.578$ ($p_{\text{raw}} = 0.0573$), nearing statistical significance. Its nonlinear temporal gates capture short-run momentum and policy rate anticipations more effectively than linear models. Under FDR control at $\alpha = 0.05$, $q_{\text{horiz}} = 0.2292$ (fails to reject equal accuracy).
- **$h=20$ (Monthly)**: VECM (6-variable) achieves $CW = 1.973$ ($p_{\text{raw}} = 0.0242$), demonstrating raw unadjusted outperformance over Naïve. However, after Benjamini-Hochberg FDR multiplicity control at the committed $\alpha = 0.05$ threshold, $q_{\text{horiz}} = 0.0968 > 0.05$ and $q_{\text{global}} = 0.2904$. It fails to reject equal accuracy at the confirmatory 5% standard (though it clears an exploratory 10% FDR threshold).
- **Headline Finding**: At the pre-committed $\alpha = 0.05$ standard with multiplicity control, **no model significantly beats the Naïve Random Walk benchmark across any horizon**.

### 3. Resolution of the Cumulative-Sum Scoring Caveat (#68)
As documented in Issue #68, iterating differenced forecasts $\hat{s}_{t+h} = s_t + \sum_{k=1}^h \widehat{\Delta s}_{t+k}$ accumulates $h \times \sigma^2_{\text{param}}$ estimation noise at $h=20$ for ARIMA and VAR. In raw terms, VAR-AIC has $\text{MSPE} = 0.015832 > 0.015639$. The Clark-West adjustment term explicitly removes this accumulated estimation variance ($0.000949$), revealing that the underlying conditional expectation produces an adjusted MSPE of $0.014883 < 0.015639$ ($CW = 0.970, p = 0.1660$). Clark-West thus provides the mathematically required correction for evaluating nested iterated forecasts.